In [ ]:
import pandas as pd
import ollama

In [ ]:
df = pd.read_pickle('dataset_binarios_solgel_solstate_18-09-26.pkl')

In [ ]:
def ask_ollama(formula, synthesis_type, model, temperature):
    if pd.isna(formula):
        return "no"

    if synthesis_type == "solid-state":
        question = (
            f"Can the compound {formula} be synthesized by solid-state synthesis? "
            "Answer only 'solid-state' if yes, or 'no' if no."
        )
        valid_answers = ["solid-state", "no"]

    elif synthesis_type == "sol-gel":
        question = (
            f"Can the compound {formula} be synthesized by sol-gel synthesis? "
            "Answer only 'sol-gel' if yes, or 'no' if no."
        )
        valid_answers = ["sol-gel", "no"]

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": question
            }
        ],
        options={
            "temperature": temperature
        }
    )

    answer = response["message"]["content"].strip().lower()

    # Normalização para garantir somente as respostas desejadas
    for valid_answer in valid_answers:
        if valid_answer in answer:
            return valid_answer
    return "invalid"


## Inferências

In [ ]:
MODEL = "qwen2.5:7b"
TEMPERATURE = 0

df["llm_solstate_qwenbase"] = df["target_formula"].apply(lambda x: ask_ollama(x, "solid-state", MODEL, TEMPERATURE))
df["llm_solgel_qwenbase"] = df["target_formula"].apply(lambda x: ask_ollama(x, "sol-gel", MODEL, TEMPERATURE))

In [ ]:
MODEL= 'hf.co/Kylan12/qwen-chemistry:Q4_K_M'
TEMPERATURE = 0

df["llm_solstate_qwenchem"] = df["target_formula"].apply(lambda x: ask_ollama(x, "solid-state", MODEL, TEMPERATURE))
df["llm_solgel_qwenchem"] = df["target_formula"].apply(lambda x: ask_ollama(x, "sol-gel", MODEL, TEMPERATURE))

In [ ]:
MODEL= 'hf.co/juntaoyuan/chemistry-assistant-7b:Q5_K_M'
TEMPERATURE = 0

df["llm_solstate_chemass"] = df["target_formula"].apply(lambda x: ask_ollama(x, "solid-state", MODEL, TEMPERATURE))
df["llm_solgel_chemass"] = df["target_formula"].apply(lambda x: ask_ollama(x, "sol-gel", MODEL, TEMPERATURE))

In [ ]:
df.to_pickle('dataset_results_slm.pkl')